# Autonomous Subagent Workflows for Senior Developers

## What you will build

You will build the workflow behind a migrate command in a coding assistant that works on a web
shop's customer database. The developer types the command, and the assistant hands the whole job to
a **subagent**, which is a second agent run with its own context, so the noise of its work never
reaches the first. The subagent applies the migration, reads every line of its log, and hands back
a short report.

Without that split, every log line lands in the developer's main session and is sent to the model
again on every later turn. The diagram shows the three mistakes this course stops: a session
filled with migration logs, a report that your code cannot read, and a report that leaves out what
the developer asks next.

![What you will build](images/subagent-overview.svg)

## Step 0: Set up the client and the model

Every call in this notebook goes through the repository's own client. Without an API key it replays
responses recorded from real runs, so you can follow the whole course for free, and with a key it
calls the model live.

In [1]:
import json
import sqlite3
from dataclasses import dataclass, replace
from typing import Literal

from pydantic import BaseModel, Field, ValidationError

from vault import get_client, load_env, model_for

load_env()
client = get_client("05-subagent-delegation/01-delegate-a-migration-to-a-subagent")
MODEL = model_for("default")

print(f"Client ready. Every request in this notebook uses {MODEL}.")

Client ready. Every request in this notebook uses google/gemini-2.5-flash-lite.


## Step 1: Create the customer database and its migrations

A migration workflow needs a real database to change, so we start with a small one held in memory.
A **schema** is the written shape of the data a database allows, meaning its tables, columns and
indexes, and a **migration** is a named script that changes that shape.

In [2]:
def create_customer_database():
    """A fresh database in memory: customers, plus a table of finished migrations."""
    database = sqlite3.connect(":memory:", isolation_level=None)
    database.executescript("""
        CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, email TEXT);
        CREATE TABLE schema_migrations (name TEXT PRIMARY KEY, rows_changed INTEGER);
    """)
    rows = [(f"Customer {i}", f"User{i}@Example.com" if i % 7 == 0 else f"user{i}@example.com")
            for i in range(1, 2001)]
    rows[1376] = ("Customer 1377", "USER42@example.com")   # the same address as customer 42
    database.executemany("INSERT INTO customers (name, email) VALUES (?, ?)", rows)
    return database


DATABASE = create_customer_database()
print(f"{DATABASE.execute('SELECT count(*) FROM customers').fetchone()[0]} customers loaded")

2000 customers loaded


The team has two migrations waiting. The first adds a lower case copy of every email address with an
index on it, and the second makes that copy unique. `BACKFILL` is our own step, which fills a new
column in batches the way a production runner does, so no single update locks the whole table.

In [3]:
MIGRATIONS = {
    "0007_add_email_lower": [
        "ALTER TABLE customers ADD COLUMN email_lower TEXT",
        "BACKFILL customers.email_lower = lower(email)",
        "CREATE INDEX idx_customers_email_lower ON customers (email_lower)",
    ],
    "0008_unique_email_lower": [
        "BACKFILL customers.email_lower = lower(trim(email))",
        "CREATE UNIQUE INDEX idx_customers_email_unique ON customers (email_lower)",
    ],
}
BATCH_SIZE = 50

print(f"{len(MIGRATIONS)} migrations waiting: {list(MIGRATIONS)}")

2 migrations waiting: ['0007_add_email_lower', '0008_unique_email_lower']


`backfill_column` updates one batch of rows at a time and writes a log line for every batch, which
is where the heavy logs in this course come from.

In [4]:
def backfill_column(statement, log):
    """Fill a column in batches, logging every batch. Returns the rows changed."""
    target, expression = statement.removeprefix("BACKFILL ").split(" = ")
    table, column = target.split(".")
    total = DATABASE.execute(f"SELECT count(*) FROM {table}").fetchone()[0]
    changed = 0
    for batch, start in enumerate(range(0, total, BATCH_SIZE), start=1):
        cursor = DATABASE.execute(
            f"UPDATE {table} SET {column} = {expression} WHERE id > ? AND id <= ?",
            (start, start + BATCH_SIZE))
        changed += cursor.rowcount
        log.append(f"   batch {batch:03d} ids {start + 1:>4}-{start + BATCH_SIZE:<4} "
                   f"updated={cursor.rowcount} pages_dirty={3 + batch % 4} wal_frames={batch * 6}")
    return changed

`run_migration` applies every statement inside one transaction. It records the migration in
`schema_migrations` only when everything worked, and on any error it rolls the whole migration
back, so the database is never left half changed.

In [5]:
def run_migration(name):
    """Apply one migration in a transaction and return its full log."""
    log, rows_changed = [f"== {name}: BEGIN on customers.db"], 0
    DATABASE.execute("BEGIN")
    try:
        for statement in MIGRATIONS[name]:
            log.append(f"-> {statement}")
            if statement.startswith("BACKFILL"):
                rows_changed += backfill_column(statement, log)
            else:
                DATABASE.execute(statement)
        DATABASE.execute("INSERT INTO schema_migrations VALUES (?, ?)", (name, rows_changed))
        DATABASE.execute("COMMIT")
        log.append(f"== {name}: COMMIT")
    except sqlite3.Error as error:
        DATABASE.execute("ROLLBACK")
        log.append(f"!! {type(error).__name__}: {error}")
        log.append(f"== {name}: ROLLBACK")
    return "\n".join(log)

`show_schema` reports what the database looks like now. The next cell also runs the first migration
once on a throwaway copy, so you can see how much log a single migration writes.

In [6]:
def show_schema():
    """The customers table's columns and indexes, and every finished migration."""
    columns = [row[1] for row in DATABASE.execute("PRAGMA table_info(customers)")]
    indexes = [row[1] for row in DATABASE.execute("PRAGMA index_list(customers)")]
    finished = DATABASE.execute("SELECT name, rows_changed FROM schema_migrations").fetchall()
    return {"columns": columns, "indexes": indexes, "finished_migrations": finished}


migration_log = run_migration("0007_add_email_lower").splitlines()
print("\n".join(migration_log[:4] + ["   ..."] + migration_log[-3:]))
print(f"\none migration wrote {len(migration_log)} log lines, "
      f"{sum(len(line) + 1 for line in migration_log)} characters")
print(show_schema())

== 0007_add_email_lower: BEGIN on customers.db
-> ALTER TABLE customers ADD COLUMN email_lower TEXT
-> BACKFILL customers.email_lower = lower(email)
   batch 001 ids    1-50   updated=50 pages_dirty=4 wal_frames=6
   ...
   batch 040 ids 1951-2000 updated=50 pages_dirty=3 wal_frames=240
-> CREATE INDEX idx_customers_email_lower ON customers (email_lower)
== 0007_add_email_lower: COMMIT

one migration wrote 45 log lines, 2913 characters
{'columns': ['id', 'name', 'email', 'email_lower'], 'indexes': ['idx_customers_email_lower'], 'finished_migrations': [('0007_add_email_lower', 2000)]}


## Step 2: Register custom commands for manual actions

Some actions should run only when the developer asks for them by name, so the assistant keeps them
as commands. A **custom command** is an action the developer triggers by typing its name, such as
`/status`, and it runs the same code every time without asking the model anything.

![Register custom commands for manual actions](images/migrate-command-step-1.svg)

In [7]:
def show_schema_status(argument):
    """/status prints the schema as it is right now."""
    return json.dumps(show_schema())


def list_finished_migrations(argument):
    """/history lists every migration recorded in schema_migrations."""
    return ", ".join(f"{name} ({rows} rows)" for name, rows in show_schema()["finished_migrations"])


COMMAND_REGISTRY = {"/status": show_schema_status, "/history": list_finished_migrations}

`run_command` splits what the developer typed into a command name and an argument, then looks the
name up in `COMMAND_REGISTRY`. An unknown name is refused with the list of real commands.

In [8]:
def run_command(line):
    """Run one command typed by the developer. The model is never involved."""
    name, _, argument = line.strip().partition(" ")
    if name not in COMMAND_REGISTRY:
        return f"Unknown command {name}. Try one of {sorted(COMMAND_REGISTRY)}."
    return COMMAND_REGISTRY[name](argument)


for line in ("/status", "/history", "/drop customers"):
    print(f"{line:16} -> {run_command(line)}")

/status          -> {"columns": ["id", "name", "email", "email_lower"], "indexes": ["idx_customers_email_lower"], "finished_migrations": [["0007_add_email_lower", 2000]]}
/history         -> 0007_add_email_lower (2000 rows)
/drop customers  -> Unknown command /drop. Try one of ['/history', '/status'].


## Step 3: Package the migration as a skill with a fork flag

Applying a migration takes judgement, because someone has to read the log and decide what happened,
so we hand that job to the model as a skill. A **skill** is a packaged workflow, meaning a set of
instructions plus the only tools it may use, which the model can choose to run when a request
matches its description.

![Package the migration as a skill with a fork flag](images/migrate-command-step-2.svg)

In [9]:
TOOL_SCHEMAS = {
    "run_migration": {"type": "function", "function": {
        "name": "run_migration",
        "description": "Apply one named migration and return its log.",
        "parameters": {"type": "object", "properties": {"name": {"type": "string"}},
                       "required": ["name"]}}},
    "show_schema": {"type": "function", "function": {
        "name": "show_schema",
        "description": "Show the customers table and every finished migration.",
        "parameters": {"type": "object", "properties": {}}}},
}
TOOL_REGISTRY = {"run_migration": run_migration, "show_schema": show_schema}

print(f"tools the migration work can use: {list(TOOL_REGISTRY)}")

tools the migration work can use: ['run_migration', 'show_schema']


The `Skill` class holds everything a skill needs. The `fork` field decides where the skill runs,
and it starts as `False`, which means the skill works inside the developer's own session.

In [10]:
@dataclass(frozen=True)
class Skill:
    name: str
    description: str      # what the main model reads to decide when to use the skill
    instructions: str     # the system prompt the skill runs under
    tool_names: tuple     # the only tools the skill may call
    fork: bool = False    # True runs the skill in a fresh history of its own


MIGRATION_SKILL = Skill(
    name="migrate_schema",
    description="Apply one named migration to the customer database and report the result.",
    instructions=("You apply database migrations. Run the migration you are given with "
                  "run_migration, then reply with a short report of what happened."),
    tool_names=("run_migration", "show_schema"))

The main model cannot see Python objects, so `describe_skill_as_tool` turns a skill into a tool
description. The model reads that description and can ask for the skill with a **tool call**,
which is the model asking your code to run a named function, sent as data.

In [11]:
def describe_skill_as_tool(skill):
    """The tool description the main model reads, with one argument: the task."""
    return {"type": "function", "function": {
        "name": skill.name, "description": skill.description,
        "parameters": {"type": "object",
                       "properties": {"task": {"type": "string",
                                               "description": "The job, in one sentence."}},
                       "required": ["task"]}}}


print(json.dumps(describe_skill_as_tool(MIGRATION_SKILL)["function"], indent=1))

{
 "name": "migrate_schema",
 "description": "Apply one named migration to the customer database and report the result.",
 "parameters": {
  "type": "object",
  "properties": {
   "task": {
    "type": "string",
    "description": "The job, in one sentence."
   }
  },
  "required": [
   "task"
  ]
 }
}


## Step 4: Run the migration inside the main session

The simplest way to run a skill is to paste its instructions into the session the developer is
already using and let the model work there. We build that first, because it works, and then we
measure what it costs every later turn.

In [12]:
def execute_tool_call(tool_call):
    """Run one requested tool and return the tool message for the history."""
    arguments = json.loads(tool_call.function.arguments or "{}")
    output = TOOL_REGISTRY[tool_call.function.name](**arguments)
    content = output if isinstance(output, str) else json.dumps(output)
    return {"role": "tool", "tool_call_id": tool_call.id, "content": content}

`run_agent_loop` is the same loop as in the first course. It calls the model, runs the tools it asks
for, and returns the answer once `finish_reason` stops being `tool_calls`.

In [13]:
def run_agent_loop(history, tool_names, max_turns=6):
    """Call the model and run its tools until it answers. Every message lands in history."""
    tools = [TOOL_SCHEMAS[name] for name in tool_names]
    for turn in range(1, max_turns + 1):
        response = client.chat.completions.create(model=MODEL, max_tokens=500,
                                                  tools=tools, messages=history)
        choice = response.choices[0]
        history.append(choice.message.model_dump(exclude_none=True))
        print(f"  turn {turn}: finish_reason={choice.finish_reason}, "
              f"prompt_tokens={response.usage.prompt_tokens}")
        if choice.finish_reason != "tool_calls":
            return choice.message.content
        for tool_call in choice.message.tool_calls:
            print(f"    runs {tool_call.function.name}({tool_call.function.arguments})")
            history.append(execute_tool_call(tool_call))
    raise RuntimeError(f"no answer after {max_turns} turns")

`run_skill_in_history` adds the skill's instructions and the task to the history it is given, then
runs the loop on that same history. Here that history is the developer's main session.

In [14]:
MAIN_SYSTEM_PROMPT = ("You are a coding assistant working with a developer "
                      "on the customer database of a web shop.")


def run_skill_in_history(skill, task, history):
    """Run a skill inside an existing history. Everything it reads stays there."""
    history.append({"role": "user", "content": f"{skill.instructions}\n\nTask: {task}"})
    return run_agent_loop(history, skill.tool_names)


DATABASE = create_customer_database()
inline_history = [{"role": "system", "content": MAIN_SYSTEM_PROMPT}]
answer = run_skill_in_history(MIGRATION_SKILL, "Apply migration 0007_add_email_lower.",
                              inline_history)
print(f"\nanswer: {answer}")
print(f"main session: {len(inline_history)} messages, "
      f"{len(json.dumps(inline_history))} characters")

  turn 1: finish_reason=tool_calls, prompt_tokens=90
    runs run_migration({"name":"0007_add_email_lower"})


  turn 2: finish_reason=stop, prompt_tokens=1503

answer: The migration 0007_add_email_lower was applied successfully. This migration added a new column `email_lower` to the customers table and backfilled it with the lowercase version of the `email` column. Finally, it created an index on the new `email_lower` column.
main session: 5 messages, 3897 characters


The migration worked, and now the developer carries on talking. `ask_main_session` asks one
follow-up and returns the answer with `prompt_tokens`, the size of the history the model was sent.

In [15]:
FOLLOW_UPS = ["Which index did that migration add, and on which table? Answer in one line.",
              "How many rows did the backfill change? Answer in one line."]


def ask_main_session(history, question):
    """Ask one follow-up in the main session. Returns the answer and what the call carried."""
    history.append({"role": "user", "content": question})
    response = client.chat.completions.create(model=MODEL, max_tokens=200, messages=history)
    answer = response.choices[0].message.content or ""
    history.append({"role": "assistant", "content": answer})
    return answer, response.usage.prompt_tokens


INLINE_TOKENS = []
for question in FOLLOW_UPS:
    answer, prompt_tokens = ask_main_session(inline_history, question)
    INLINE_TOKENS.append(prompt_tokens)
    print(f"prompt_tokens={prompt_tokens:>5}  {question}\n{'':20}{answer.strip()}")

prompt_tokens= 1554  Which index did that migration add, and on which table? Answer in one line.
                    The migration added the index `idx_customers_email_lower` on the `customers` table.


prompt_tokens= 1589  How many rows did the backfill change? Answer in one line.
                    The backfill changed 2000 rows.


Each follow-up is one short question, yet the two calls were sent with 1554 and 1589 prompt tokens,
because each one carries every batch line of the migration log. The log was read once, but the main
session pays to send it again on every turn for the rest of the day.

## Step 5: Fork the skill into a subagent with its own history

The fix is to run the skill somewhere else and keep only its result. **Context forking** means
starting the skill in a fresh history of its own, seeded with nothing but its instructions and its
task, so nothing it reads can land in the main session.

![Fork the skill into a subagent with its own history](images/migrate-command-step-3.svg)

In [16]:
MAX_SUBAGENT_TURNS = 6   # a subagent that keeps asking for tools still stops


def run_subagent(skill, task):
    """Run a skill in a forked history. The caller gets the reply and that history."""
    subagent_history = [{"role": "system", "content": skill.instructions},
                        {"role": "user", "content": task}]
    reply = run_agent_loop(subagent_history, skill.tool_names, max_turns=MAX_SUBAGENT_TURNS)
    return reply, subagent_history

`run_skill` is the one place that reads the `fork` flag. A skill with `fork=True` goes to a
subagent, and the main history it was handed is never touched.

In [17]:
def run_skill(skill, task, main_history):
    """Honour the fork flag. Returns the reply and the history the work happened in."""
    if not skill.fork:
        return run_skill_in_history(skill, task, main_history), main_history
    return run_subagent(skill, task)


FORKED_MIGRATION_SKILL = replace(MIGRATION_SKILL, fork=True)
print(f"{FORKED_MIGRATION_SKILL.name}: fork={FORKED_MIGRATION_SKILL.fork}")

migrate_schema: fork=True


The next cell starts from a fresh database and runs the same migration with the forked skill, then
compares the two histories.

In [18]:
DATABASE = create_customer_database()
main_history = [{"role": "system", "content": MAIN_SYSTEM_PROMPT}]
reply, subagent_history = run_skill(FORKED_MIGRATION_SKILL,
                                    "Apply migration 0007_add_email_lower.", main_history)

print(f"\nsubagent history : {len(subagent_history)} messages, "
      f"{len(json.dumps(subagent_history))} characters")
print(f"main history     : {len(main_history)} message, {len(json.dumps(main_history))} characters")
print(f"reply            : {reply}")

  turn 1: finish_reason=tool_calls, prompt_tokens=69
    runs run_migration({"name":"0007_add_email_lower"})


  turn 2: finish_reason=stop, prompt_tokens=1482

subagent history : 5 messages, 3784 characters
main history     : 1 message, 126 characters
reply            : The migration `0007_add_email_lower` was applied successfully. This migration added a new column `email_lower` to the `customers` table, backfilled it with the lowercase version of the existing `email` column, and created an index on the new column.


The subagent's history holds 3784 characters, including the whole log, while the main history still
holds nothing but its system prompt. Only the reply is left to cross back.

## Step 6: Return a typed summary instead of prose

The reply that comes back is written for a person, but the code in the main session has to act on
it, for example to stop the next migration after a failure. So we define the exact shape the
subagent must hand back, as a Pydantic model.

![Return a typed summary instead of prose](images/migration-summary-step-1.svg)

In [19]:
class MigrationSummary(BaseModel):
    """The only thing allowed to cross back from the migration subagent."""
    migration: str
    status: Literal["applied", "rolled_back", "unknown"]
    rows_changed: int
    error: str = Field(max_length=200)
    needs_review: bool


try:
    MigrationSummary.model_validate_json(reply or "")
except ValidationError as error:
    print(f"the reply is not a summary: {error.errors()[0]['type']}")
    print(f"it starts: {(reply or '')[:70]!r}")

the reply is not a summary: json_invalid
it starts: 'The migration `0007_add_email_lower` was applied successfully. This mi'


The reply fails on its very first character, because it is prose rather than JSON. `validate_summary`
makes the boundary return one type whatever arrives: a valid summary, or one flagged for a person
with `needs_review` set.

In [20]:
def validate_summary(text, migration):
    """Always return a MigrationSummary. Text that does not validate becomes a flagged one."""
    try:
        return MigrationSummary.model_validate_json(text)
    except ValidationError as error:
        return MigrationSummary(migration=migration, status="unknown", rows_changed=0,
                                error=f"summary rejected: {error.error_count()} problems",
                                needs_review=True)


print(validate_summary(reply or "", "0007_add_email_lower"))

migration='0007_add_email_lower' status='unknown' rows_changed=0 error='summary rejected: 1 problems' needs_review=True


The other half is to ask for the right shape in the first place. The subagent gets one last turn,
still inside its own history, and `response_format` tells the model to reply as JSON that matches
the `MigrationSummary` class.

In [21]:
SUMMARY_FORMAT = {"type": "json_schema", "json_schema": {
    "name": "migration_summary", "strict": True,
    "schema": MigrationSummary.model_json_schema() | {"additionalProperties": False}}}


def summarise_subagent_run(subagent_history, migration):
    """The subagent's last turn: report as a MigrationSummary, from inside its own history."""
    subagent_history.append({"role": "user", "content": "Report the migration as a summary."})
    response = client.chat.completions.create(model=MODEL, max_tokens=300,
                                              response_format=SUMMARY_FORMAT,
                                              messages=subagent_history)
    return validate_summary(response.choices[0].message.content or "", migration)


summary = summarise_subagent_run(subagent_history, "0007_add_email_lower")
print(summary.model_dump_json(indent=1))

{
 "migration": "0007_add_email_lower",
 "status": "applied",
 "rows_changed": 2000,
 "error": "",
 "needs_review": false
}


## Step 7: Hand the migrate command to the forked skill

Now we join the command, the forked skill and the typed summary into one path. The developer types
`/migrate`, which is a manual action, and the command hands the work to the forked skill and returns
only its typed summary, so the log stays in the subagent.

![Hand the migrate command to the forked skill](images/migration-summary-step-2.svg)

In [22]:
SUMMARIES_RETURNED = []


def run_forked_migration(skill, task, migration):
    """Run a forked skill, then keep nothing but its typed summary."""
    reply, subagent_history = run_skill(skill, task, main_history=None)
    summary = summarise_subagent_run(subagent_history, migration)
    SUMMARIES_RETURNED.append(summary)
    print(f"  subagent history: {len(json.dumps(subagent_history))} characters, left behind")
    return summary


def format_handover(summary):
    """What crosses back to the main session: the typed summary, as JSON."""
    return summary.model_dump_json()


def run_migrate_command(argument):
    """/migrate <name> hands one migration to the forked skill."""
    task = f"Apply migration {argument}."
    return format_handover(run_forked_migration(FORKED_MIGRATION_SKILL, task, argument))


COMMAND_REGISTRY["/migrate"] = run_migrate_command
print(f"commands: {sorted(COMMAND_REGISTRY)}")

commands: ['/history', '/migrate', '/status']


The developer runs the same migration as in Step 4 on a fresh database, then asks the same two
follow-up questions. Only the command's output joins the main session.

In [23]:
DATABASE = create_customer_database()
main_history = [{"role": "system", "content": MAIN_SYSTEM_PROMPT}]
command_output = run_command("/migrate 0007_add_email_lower")
main_history.append({"role": "user",
                     "content": f"I ran /migrate 0007_add_email_lower. Result: {command_output}"})
print(f"\ncommand output: {command_output}\n")

FORKED_TOKENS = []
for question in FOLLOW_UPS:
    answer, prompt_tokens = ask_main_session(main_history, question)
    FORKED_TOKENS.append(prompt_tokens)
    print(f"prompt_tokens={prompt_tokens:>5}  {question}\n{'':20}{answer.strip()}")

  turn 1: finish_reason=tool_calls, prompt_tokens=69
    runs run_migration({"name":"0007_add_email_lower"})


  turn 2: finish_reason=stop, prompt_tokens=1482


  subagent history: 3887 characters, left behind

command output: {"migration":"0007_add_email_lower","status":"applied","rows_changed":2000,"error":"","needs_review":false}



prompt_tokens=   89  Which index did that migration add, and on which table? Answer in one line.
                    The migration `0007_add_email_lower` added an index on the `email_lower` column of the `customers` table.


prompt_tokens=  135  How many rows did the backfill change? Answer in one line.
                    The backfill changed 2000 rows.


The table below compares the two runs turn by turn. The work was identical, and only the history it
happened in has changed.

In [24]:
print(f"{'follow-up':10} {'inline':>8} {'forked':>8}")
for number, (inline, forked) in enumerate(zip(INLINE_TOKENS, FORKED_TOKENS), start=1):
    print(f"{number:<10} {inline:>8} {forked:>8}")
print(f"\nthe forked main session carries {sum(FORKED_TOKENS)} prompt tokens "
      f"across both follow-ups, against {sum(INLINE_TOKENS)} inline")

follow-up    inline   forked
1              1554       89
2              1589      135

the forked main session carries 224 prompt tokens across both follow-ups, against 3143 inline


The forked session carried 224 prompt tokens across both follow-ups, against 3143 inline, and it
answered the row count correctly from the summary. Read the first answer again, though. The inline
session named the index `idx_customers_email_lower`, while the forked one could only say that an
index was added on `email_lower`, because the summary never named it, and Step 9 comes back to that.

## Step 8: Let the model choose the skill from a plain request

A command is for when the developer knows exactly what to run, but often they simply describe what
they want. The main model then reads the skill's description and decides by itself to call it,
and the call still goes to a forked subagent.

![Let the model choose the skill from a plain request](images/migrate-command-step-4.svg)

In [25]:
SKILL_REGISTRY = {FORKED_MIGRATION_SKILL.name: FORKED_MIGRATION_SKILL}
SKILL_TOOLS = [describe_skill_as_tool(skill) for skill in SKILL_REGISTRY.values()]


def execute_skill_call(tool_call):
    """Run the requested skill in a fork and send back only its handover."""
    name = tool_call.function.name
    print(f"  main model calls {name}({tool_call.function.arguments})")
    if name not in SKILL_REGISTRY:
        content = json.dumps({"error": f"Unknown skill {name}. Skills: {sorted(SKILL_REGISTRY)}"})
    else:
        task = json.loads(tool_call.function.arguments)["task"]
        content = format_handover(run_forked_migration(SKILL_REGISTRY[name], task, task))
    return {"role": "tool", "tool_call_id": tool_call.id, "content": content}

`answer_developer_request` is the main session's own loop. Its only tools are skills, so anything
heavy the model wants to do happens in a subagent.

In [26]:
def answer_developer_request(history, request, max_turns=4):
    """The main session's loop. Every tool it can call is a forked skill."""
    history.append({"role": "user", "content": request})
    for turn in range(1, max_turns + 1):
        response = client.chat.completions.create(model=MODEL, max_tokens=300,
                                                  tools=SKILL_TOOLS, messages=history)
        choice = response.choices[0]
        history.append(choice.message.model_dump(exclude_none=True))
        print(f"main turn {turn}: finish_reason={choice.finish_reason}")
        if choice.finish_reason != "tool_calls":
            return choice.message.content, response.usage.prompt_tokens
        for tool_call in choice.message.tool_calls:
            history.append(execute_skill_call(tool_call))
    raise RuntimeError(f"no answer after {max_turns} turns")


MAKE_UNIQUE_REQUEST = "Now make customer emails unique with migration 0008_unique_email_lower."
turns_before = len(main_history)
answer, prompt_tokens = answer_developer_request(main_history, MAKE_UNIQUE_REQUEST)
print(f"\nanswer: {answer}")
print(f"skill result the model saw: {[m for m in main_history if m['role'] == 'tool'][-1]['content']}")

main turn 1: finish_reason=tool_calls
  main model calls default_api.migrate_schema({"task":"Apply 0008_unique_email_lower migration to make customer emails unique."})


main turn 2: finish_reason=stop

answer: I am sorry, I cannot fulfill this request. The available tool `migrate_schema` does not support the task of making customer emails unique with migration 0008_unique_email_lower.
skill result the model saw: {"error": "Unknown skill default_api.migrate_schema. Skills: ['migrate_schema']"}


The main model did choose the skill, but it asked for `default_api.migrate_schema`, a name that
starts with a namespace we never declared. The registry refused it, so the model told the developer
it could not do the job, and the migration never ran.

`find_skill` accepts a registered name, or the part after the last dot when that part is itself a
registered skill. A namespace the model adds is forgiven, but no skill outside the registry can
ever run.

In [27]:
def find_skill(name):
    """Match a registered skill, allowing a namespace prefix the model added, such as a.b."""
    if name in SKILL_REGISTRY:
        return SKILL_REGISTRY[name]
    return SKILL_REGISTRY.get(name.rsplit(".", 1)[-1])


def execute_skill_call(tool_call):
    """Run the requested skill in a fork. A name that matches no skill is refused."""
    name, skill = tool_call.function.name, find_skill(tool_call.function.name)
    print(f"  main model calls {name}({tool_call.function.arguments})")
    if skill is None:
        content = json.dumps({"error": f"Unknown skill {name}. Skills: {sorted(SKILL_REGISTRY)}"})
    else:
        task = json.loads(tool_call.function.arguments)["task"]
        content = format_handover(run_forked_migration(skill, task, task))
    return {"role": "tool", "tool_call_id": tool_call.id, "content": content}


print(f"default_api.migrate_schema -> {find_skill('default_api.migrate_schema').name}")
print(f"drop_database              -> {find_skill('drop_database')}")

default_api.migrate_schema -> migrate_schema
drop_database              -> None


The next cell removes the refused attempt from the main history and sends the same request again.

In [28]:
del main_history[turns_before:]   # drop the refused attempt, as if it never happened
answer, prompt_tokens = answer_developer_request(main_history, MAKE_UNIQUE_REQUEST)
print(f"\nprompt_tokens={prompt_tokens}\nanswer: {answer}")
print(f"skill result the model saw: {[m for m in main_history if m['role'] == 'tool'][-1]['content']}")

main turn 1: finish_reason=tool_calls
  main model calls default_api.migrate_schema({"task":"Apply migration 0008_unique_email_lower to make customer emails unique."})


  turn 1: finish_reason=tool_calls, prompt_tokens=74
    runs run_migration({"name":"0008_unique_email_lower"})


  turn 2: finish_reason=stop, prompt_tokens=1494


  subagent history: 4062 characters, left behind


main turn 2: finish_reason=stop

prompt_tokens=259
answer: The migration `0008_unique_email_lower` failed because the `email_lower` column in the `customers` table has duplicate entries. This needs to be resolved before the migration can be applied.
skill result the model saw: {"migration":"0008_unique_email_lower","status":"rolled_back","rows_changed":0,"error":"UNIQUE constraint failed: customers.email_lower","needs_review":true}


The model sent the same namespaced name again, and this time it resolved to the migration skill. The
subagent ran migration 0008, which hit the duplicate address and rolled back. The main session got
only the summary, with `rolled_back` and `needs_review` set, so it told the developer about the
duplicates instead of carrying on.

## Step 9: Hand back the facts the next question needs

A summary is the only thing that crosses back, so it has to carry the facts the main session will
be asked about next. In Step 7 the forked session could not name the new index because the summary
never mentioned it, so we take facts like that from the database instead of asking the model.

![Hand back the facts the next question needs](images/migration-summary-step-3.svg)

In [29]:
def check_summary_against_database(summary):
    """List every claim in the summary that the migration table contradicts."""
    row = DATABASE.execute("SELECT rows_changed FROM schema_migrations WHERE name = ?",
                           (summary.migration,)).fetchone()
    problems = []
    if row is None and summary.status == "applied":
        problems.append(f"says applied, but {summary.migration} is not in schema_migrations")
    if row is not None and summary.status != "applied":
        problems.append(f"says {summary.status}, but schema_migrations has it")
    if row is not None and row[0] != summary.rows_changed:
        problems.append(f"says {summary.rows_changed} rows changed, the table says {row[0]}")
    return problems


for summary in SUMMARIES_RETURNED:
    print(f"{summary.migration}: {check_summary_against_database(summary) or 'matches the table'}")

0007_add_email_lower: matches the table
0008_unique_email_lower: matches the table


Both summaries match `schema_migrations` in this run, so the model reported the status and the row
count correctly. `format_handover` now does two jobs before anything crosses back. It flags a
summary that the migration table contradicts, and it attaches the schema as the database reports
it, so the main session never has to trust the model for a table or index name.

In [30]:
def format_handover(summary):
    """The checked summary, plus the schema read from the database rather than the model."""
    problems = check_summary_against_database(summary)
    if problems:
        summary = summary.model_copy(update={"needs_review": True,
                                             "error": "; ".join(problems)[:200]})
    return json.dumps({"summary": summary.model_dump(), "schema_after": show_schema()})


print(format_handover(SUMMARIES_RETURNED[0]))

{"summary": {"migration": "0007_add_email_lower", "status": "applied", "rows_changed": 2000, "error": "", "needs_review": false}, "schema_after": {"columns": ["id", "name", "email", "email_lower"], "indexes": ["idx_customers_email_lower"], "finished_migrations": [["0007_add_email_lower", 2000]]}}


Migration 0008 was rolled back, so the database is exactly as 0007 left it. The next cell hands the
0007 summary over in its new form and asks the index question from Step 7 again.

In [31]:
checked_history = [{"role": "system", "content": MAIN_SYSTEM_PROMPT},
                   {"role": "user", "content": "I ran /migrate 0007_add_email_lower. "
                                               f"Result: {format_handover(SUMMARIES_RETURNED[0])}"}]
answer, prompt_tokens = ask_main_session(checked_history, FOLLOW_UPS[0])
print(f"prompt_tokens={prompt_tokens:>5}  {FOLLOW_UPS[0]}\n{'':20}{answer.strip()}")

prompt_tokens=  159  Which index did that migration add, and on which table? Answer in one line.
                    The migration added the index `idx_customers_email_lower` on the `customers` table.


The answer now names `idx_customers_email_lower` on `customers`, as the inline session did, and the
call carried 159 prompt tokens against 1554 inline. The log still never left the subagent.

## Step 10: Test the delegation without calling the model

Each safeguard above gets a test that runs in milliseconds with no API key, so it can run on every
commit. If someone runs the skill inline again, lets prose through the boundary or drops the
database check, one of these tests fails.

![Test the delegation without calling the model](images/migration-summary-step-4.svg)

In [32]:
def test_no_log_line_reaches_the_main_session():
    assert "batch 001" in json.dumps(inline_history), "the inline run should carry the log"
    for history in (main_history, checked_history):
        assert "batch 001" not in json.dumps(history), "a log line is in the main session"


def test_prose_becomes_a_flagged_summary():
    summary = validate_summary("The migration went fine.", "0007_add_email_lower")
    assert isinstance(summary, MigrationSummary) and summary.needs_review


def test_unknown_command_is_refused():
    assert run_command("/drop customers").startswith("Unknown command")


def test_skill_names_resolve_only_to_registered_skills():
    assert find_skill("default_api.migrate_schema") is FORKED_MIGRATION_SKILL
    assert find_skill("drop_database") is None

The last two tests hand over a summary with a wrong row count, and check that the table catches it
and that the real index name crosses back.

In [33]:
WRONG_SUMMARY = MigrationSummary(migration="0007_add_email_lower", status="applied",
                                 rows_changed=0, error="", needs_review=False)


def test_wrong_row_count_is_flagged():
    assert json.loads(format_handover(WRONG_SUMMARY))["summary"]["needs_review"]


def test_handover_names_the_real_index():
    assert "idx_customers_email_lower" in format_handover(WRONG_SUMMARY)


for test in (test_no_log_line_reaches_the_main_session, test_prose_becomes_a_flagged_summary,
             test_unknown_command_is_refused, test_skill_names_resolve_only_to_registered_skills,
             test_wrong_row_count_is_flagged, test_handover_names_the_real_index):
    test()
    print(f"passed: {test.__name__}")
DATABASE.close()

passed: test_no_log_line_reaches_the_main_session
passed: test_prose_becomes_a_flagged_summary
passed: test_unknown_command_is_refused
passed: test_skill_names_resolve_only_to_registered_skills
passed: test_wrong_row_count_is_flagged
passed: test_handover_names_the_real_index


## Concepts

| Concept | Where it lives | What it does |
|---|---|---|
| **Custom command** | `COMMAND_REGISTRY` and `run_command` | Runs a manual action the developer names, with no model involved |
| **Skill** | `Skill` and `describe_skill_as_tool` | Packages instructions and tools that the model can choose to run |
| **Fork flag** | `Skill.fork`, read only by `run_skill` | Decides whether a skill works in the main session or in a subagent |
| **Context forking** | `run_subagent` | Starts the skill in a fresh history, so its logs never reach the main session |
| **Subagent delegation** | `run_forked_migration` | Hands the whole job to a subagent and keeps only what it returns |
| **Typed summary** | `MigrationSummary`, `validate_summary` | The one shape allowed back, with prose turned into a flagged summary |
| **Checked handover** | `format_handover` | Checks the summary against the migration table and attaches the real schema |